In [1]:
import pandas as pd
from tqdm import tqdm

def create_long_df(df):
    """
    Convert the game-level DataFrame into a long format where each row corresponds 
    to a team’s performance in a game.
    """
    # Process winner rows
    winners = df.copy()
    winners['TeamID'] = winners['WTeamID']
    winners['Score'] = winners['WScore']
    winners['FGM']   = winners['WFGM']
    winners['FGA']   = winners['WFGA']
    winners['FGM3']  = winners['WFGM3']
    winners['FGA3']  = winners['WFGA3']
    winners['FTM']   = winners['WFTM']
    winners['FTA']   = winners['WFTA']
    winners['OR']    = winners['WOR']
    winners['DR']    = winners['WDR']
    winners['Ast']   = winners['WAst']
    winners['TO']    = winners['WTO']
    winners['Stl']   = winners['WStl']
    winners['Blk']   = winners['WBlk']
    winners['PF']    = winners['WPF']
    # For winners, location is as given
    winners['Loc']   = winners['WLoc']
    winners['is_winner'] = 1

    # Process loser rows
    losers = df.copy()
    losers['TeamID'] = losers['LTeamID']
    losers['Score'] = losers['LScore']
    losers['FGM']   = losers['LFGM']
    losers['FGA']   = losers['LFGA']
    losers['FGM3']  = losers['LFGM3']
    losers['FGA3']  = losers['LFGA3']
    losers['FTM']   = losers['LFTM']
    losers['FTA']   = losers['LFTA']
    losers['OR']    = losers['LOR']
    losers['DR']    = losers['LDR']
    losers['Ast']   = losers['LAst']
    losers['TO']    = losers['LTO']
    losers['Stl']   = losers['LStl']
    losers['Blk']   = losers['LBlk']
    losers['PF']    = losers['LPF']
    # Invert location: if winning team was at home, then loser is away and vice-versa.
    losers['Loc'] = losers['WLoc'].map(lambda x: 'A' if x=='H' else ('H' if x=='A' else 'N'))
    losers['is_winner'] = 0

    common_cols = ['Season', 'DayNum', 'TeamID', 'Score', 'FGM', 'FGA', 
                   'FGM3', 'FGA3', 'FTM', 'FTA', 'OR', 'DR', 'Ast', 'TO', 
                   'Stl', 'Blk', 'PF', 'Loc', 'is_winner']
    
    long_df = pd.concat([winners[common_cols], losers[common_cols]], ignore_index=True)
    return long_df

def compute_rolling_stats(long_df, n_matches=5):
    """
    Compute the rolling (previous n_matches) average for each statistic for each team in a season.
    A shift of 1 is applied so that the current game is not included.
    Also computes the rolling averages for one-hot encoded location indicators.
    """
    # Define the stat columns to average
    stat_cols = ['Score', 'FGM', 'FGA', 'FGM3', 'FGA3', 
                 'FTM', 'FTA', 'OR', 'DR', 'Ast', 'TO', 'Stl', 'Blk', 'PF']
    
    # Create one-hot columns for location
    long_df['home']    = (long_df['Loc'] == 'H').astype(int)
    long_df['away']    = (long_df['Loc'] == 'A').astype(int)
    long_df['neutral'] = (long_df['Loc'] == 'N').astype(int)
    onehot_cols = ['home', 'away', 'neutral']
    
    # Sort by TeamID, Season, and DayNum so the rolling window is correct.
    long_df = long_df.sort_values(by=['TeamID', 'Season', 'DayNum'])
    
    # Compute rolling average for each statistic (shift to use only previous games)
    for col in stat_cols:
        long_df[col + '_avg'] = long_df.groupby(['TeamID', 'Season'])[col] \
                                        .transform(lambda x: x.shift(1).rolling(n_matches, min_periods=n_matches).mean())
    for col in onehot_cols:
        long_df[col + '_avg'] = long_df.groupby(['TeamID', 'Season'])[col] \
                                        .transform(lambda x: x.shift(1).rolling(n_matches, min_periods=n_matches).mean())
    return long_df

def create_training_data_vectorized(df, start_year, n_matches=5, n_seasons=3):
    """
    Create training data (X and y) using vectorized operations.

    Process only games from seasons >= (start_year - n_seasons).
    For each game, obtain the rolling averages (computed from previous n_matches)
    for both the winner and loser, then create two samples:
      - [winner_rolling | loser_rolling] with label 1 (team 1 wins)
      - [loser_rolling | winner_rolling] with label 0 (team 1 loses)

    Returns:
    - X_df (pd.DataFrame): Each row is the concatenated rolling features for team1 and team2.
    - y_df (pd.Series): The corresponding target (1 if team1 wins, 0 otherwise).
    """
    # Filter the games to those in seasons we're interested in.
    df_filtered = df[df['Season'] >= start_year - n_seasons].copy()
    print("copy created")
    
    # Create a long-format DataFrame (one row per team per game)
    long_df = create_long_df(df_filtered)

    print("long_df created")
    
    # Compute rolling averages for each team using vectorized groupby operations.
    long_df = compute_rolling_stats(long_df, n_matches=n_matches)

    print("rolling stats computed")
    
    # We'll extract the rolling features from the long_df.
    # Choose the columns that were created by compute_rolling_stats.
    rolling_cols = [col for col in long_df.columns if col.endswith('_avg')]

    print("rolling cols selected")
    
    # Merge the rolling averages back into the original game-level DataFrame.
    # For the winner:
    df_winner = df_filtered[['Season', 'DayNum', 'WTeamID']].copy()
    df_winner = df_winner.rename(columns={'WTeamID': 'TeamID'})
    df_winner = df_winner.merge(long_df[['Season', 'DayNum', 'TeamID'] + rolling_cols],
                                on=['Season', 'DayNum', 'TeamID'], how='left')
    # Rename columns to indicate winner stats.
    df_winner = df_winner.rename(columns={col: 'W_' + col for col in rolling_cols})

    print("winner stats merged")
    
    # For the loser:
    df_loser = df_filtered[['Season', 'DayNum', 'LTeamID']].copy()
    df_loser = df_loser.rename(columns={'LTeamID': 'TeamID'})
    df_loser = df_loser.merge(long_df[['Season', 'DayNum', 'TeamID'] + rolling_cols],
                              on=['Season', 'DayNum', 'TeamID'], how='left')
    df_loser = df_loser.rename(columns={col: 'L_' + col for col in rolling_cols})

    print("loser stats merged")
    
    # For the winner rolling stats:
    df_winner = df_filtered[['Season', 'DayNum', 'WTeamID']].copy()
    df_winner = df_winner.rename(columns={'WTeamID': 'TeamID'})
    df_winner = df_winner.merge(long_df[['Season', 'DayNum', 'TeamID'] + rolling_cols],
                                on=['Season', 'DayNum', 'TeamID'], how='left')
    df_winner = df_winner.rename(columns={col: 'W_' + col for col in rolling_cols})
    # Merge winner stats back into the main DataFrame using the correct key.
    df_merged = df_filtered.merge(df_winner, left_on=['Season', 'DayNum', 'WTeamID'],
                                right_on=['Season', 'DayNum', 'TeamID'], how='left')
    # Drop the extra 'TeamID' column from the merge.
    df_merged.drop(columns=['TeamID'], inplace=True)

    # For the loser rolling stats:
    df_loser = df_filtered[['Season', 'DayNum', 'LTeamID']].copy()
    df_loser = df_loser.rename(columns={'LTeamID': 'TeamID'})
    df_loser = df_loser.merge(long_df[['Season', 'DayNum', 'TeamID'] + rolling_cols],
                            on=['Season', 'DayNum', 'TeamID'], how='left')
    df_loser = df_loser.rename(columns={col: 'L_' + col for col in rolling_cols})
    # Merge loser stats using the losing team key.
    df_merged = df_merged.merge(df_loser, left_on=['Season', 'DayNum', 'LTeamID'],
                                right_on=['Season', 'DayNum', 'TeamID'], how='left')
    df_merged.drop(columns=['TeamID'], inplace=True)

    
    print("merged")

    # Drop games where either team does not have enough prior matches.
    required_cols = [col for col in df_merged.columns if col.endswith('_avg')]
    df_merged = df_merged.dropna(subset=required_cols)
    
    print("dropped NaNs")

    # Now, create training samples.
    X_rows = []
    y_rows = []
    
    # List of feature columns for winner and loser (order is important).
    w_features = [col for col in df_merged.columns if col.startswith('W_') and col.endswith('_avg')]
    l_features = [col for col in df_merged.columns if col.startswith('L_') and col.endswith('_avg')]
    
    # Use tqdm to show progress while iterating over the merged DataFrame.
    for _, row in tqdm(df_merged.iterrows(), total=len(df_merged), desc="Building training samples"):
        # Sample 1: team 1 is winner, team 2 is loser, label 1
        sample1 = pd.concat([row[w_features], row[l_features]])
        X_rows.append(sample1)
        y_rows.append(1)
        # Sample 2: team 1 is loser, team 2 is winner, label 0
        sample2 = pd.concat([row[l_features], row[w_features]])
        X_rows.append(sample2)
        y_rows.append(0)
    
    X_df = pd.DataFrame(X_rows)
    y_df = pd.Series(y_rows, name='target')
    
    return X_df, y_df




In [2]:
df = pd.read_csv("../../data/MRegularSeasonDetailedResults.csv")

In [3]:
start_year = 2019
X, y = create_training_data_vectorized(df, start_year, n_matches=5, n_seasons=3)
print(X.head())
print(y.head())

copy created
long_df created
rolling stats computed
rolling cols selected
winner stats merged
loser stats merged
merged
dropped NaNs


Building training samples: 100%|██████████| 41993/41993 [00:37<00:00, 1117.53it/s]


     W_Score_avg  W_FGM_avg  W_FGA_avg  W_FGM3_avg  W_FGA3_avg  W_FTM_avg  \
580         86.6       30.8       62.0         9.6        28.8       15.4   
580         86.6       30.8       62.0         9.6        28.8       15.4   
691         79.2       27.4       61.6         4.6        15.8       19.8   
691         79.2       27.4       61.6         4.6        15.8       19.8   
730         89.8       30.4       62.0         9.2        22.6       19.8   

     W_FTA_avg  W_OR_avg  W_DR_avg  W_Ast_avg  ...  L_OR_avg  L_DR_avg  \
580       22.4       9.0      24.0       19.2  ...      11.0      24.4   
580       22.4       9.0      24.0       19.2  ...      11.0      24.4   
691       29.0      16.8      27.8       10.6  ...      13.4      28.6   
691       29.0      16.8      27.8       10.6  ...      13.4      28.6   
730       29.0      14.0      29.8       19.6  ...      15.8      28.2   

     L_Ast_avg  L_TO_avg  L_Stl_avg  L_Blk_avg  L_PF_avg  L_home_avg  \
580        9.6      